In [1]:
# Mount Google Drive (optional for saving files)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install required librariess
!pip install transformers torch flask flask-ngrok pdfplumber keybert nltk fastapi "uvicorn[standard]" aiosmtplib

# Download NLTK data
import nltk
nltk.download('punkt')

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 19.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load DialoGPT (small) for conversational response generation
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Function to get chatbot response
def get_response(user_input, chat_history_ids=None):
    # Encode input and append chat history
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Generate response
    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response, chat_history_ids


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
import pdfplumber
from keybert import KeyBERT
from transformers import pipeline

# Initialize summarizer and keyword extractor
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
kw_model = KeyBERT()

# Extract text from uploaded PDF file
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

# Summarize document text
def summarize_text(text):
    summary = summarizer(text, max_length=150, min_length=40, do_sample=False)
    return summary[0]['summary_text']

# Extract keywords
def extract_keywords(text, num_keywords=10):
    keywords = kw_model.extract_keywords(text, top_n=num_keywords)
    return [kw[0] for kw in keywords]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
import secrets
secret_key = secrets.token_hex(16)
print(secret_key)


3e525f1b5e8aac2ab312793d4201ea14


In [11]:
!ngrok authtoken 33uNM1mtRSXOYOjraapnSurcFKL_4FSVtgDYjKUZEsrVaoucN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [13]:
faq_content = """
[
  {"question": "What is the leave policy?", "answer": "Annual leave is 20 days per year with prior approval."},
  {"question": "How to reset my IT password?", "answer": "You can reset your password using the self-service portal."},
  {"question": "When is the next company event?", "answer": "The next event is the annual picnic scheduled for December 10th."},
  {"question": "Who should I contact for payroll issues?", "answer": "Please contact the payroll department at payroll@company.com."},
  {"question": "How do I request work from home?", "answer": "Submit your WFH request through the HR portal. Approval is usually within 2 business days."},
  {"question": "How can I find my employee ID?", "answer": "Your employee ID is printed on your ID card and also listed in your HR profile online."},
  {"question": "What are the IT support hours?", "answer": "IT support is available from 9am to 6pm, Monday to Friday."},
  {"question": "How do I apply for medical insurance?", "answer": "You can apply for medical insurance by filling out the form on the HR portal under 'Benefits'."},
  {"question": "Can I update my personal information?", "answer": "Yes, you can update your information directly in your HR portal under 'My Profile'."},
  {"question": "Who do I contact for building maintenance?", "answer": "Email facilities@company.com for any maintenance requests."},
  {"question": "What is the company's dress code?", "answer": "Our dress code is smart casual from Monday to Thursday, and casual on Fridays."},
  {"question": "How to report a harassment incident?", "answer": "Report incidents confidentially via the HR helpdesk or email hr-confidential@company.com."},
  {"question": "Is meal service available at office?", "answer": "Yes, cafeteria service is available from 8am to 4pm at all locations."},
  {"question": "Hello", "answer": "Hi there! How can I help you today?"},
  {"question": "Hi", "answer": "Hello! Is there something I can assist you with?"},
  {"question": "Good morning", "answer": "Good morning! Hope you have a great day. What would you like to know?"},
  {"question": "Good afternoon", "answer": "Good afternoon! How can I help you today?"},
  {"question": "Thank you", "answer": "You're welcome! Let me know if you need anything else."},
  {"question": "Bye", "answer": "Goodbye! Have a wonderful day."},
  {"question": "How are you?", "answer": "I'm just a chatbot, but I'm here to help you!"},
  {"question": "Who created you?", "answer": "I was created by our IT and HR teams to assist employees."},
  {"question": "What can you do?", "answer": "I can answer questions about HR policies, IT support, company events, and more. You can also upload documents for me to analyze."}
]
"""

with open('faq_data.json', 'w') as f:
    f.write(faq_content)


In [14]:
import json
with open('faq_data.json') as f:
    faq_data = json.load(f)


In [15]:
!pip install sentence-transformers


In [16]:
from sentence_transformers import SentenceTransformer, util
import torch

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Extract questions from FAQ
questions = [item['question'] for item in faq_data]

# Pre-compute embeddings for FAQ questions
question_embeddings = embedder.encode(questions, convert_to_tensor=True)


In [17]:
def get_best_answer(user_query):
    query_embedding = embedder.encode(user_query, convert_to_tensor=True)
    cos_scores = util.pytorch_cos_sim(query_embedding, question_embeddings)[0]
    top_result = torch.topk(cos_scores, k=1)
    top_idx = top_result.indices.item()
    top_score = top_result.values.item()
    if top_score > 0.6:  # similarity threshold
        return faq_data[top_idx]['answer']
    else:
        return "Sorry, I do not have information on that. Please contact HR or IT support."


In [18]:
user_input = "How do I reset my password?"
print(get_best_answer(user_input))


You can reset your password using the self-service portal.


In [20]:
import json
with open('faq_data.json') as f:
    faq_data = json.load(f)

# Re-create embedding and questions list
questions = [item['question'] for item in faq_data]
question_embeddings = embedder.encode(questions, convert_to_tensor=True)

def get_best_answer(user_query):
    query_embedding = embedder.encode(user_query, convert_to_tensor=True)
    cos_scores = util.pytorch_cos_sim(query_embedding, question_embeddings)[0]
    top_result = torch.topk(cos_scores, k=1)
    top_idx = top_result.indices.item()
    top_score = top_result.values.item()
    if top_score > 0.5:  # Try lowering the threshold
        return faq_data[top_idx]['answer']
    else:
        return "Sorry, I do not have information on that. Please contact HR or IT support."


In [ ]:
from flask import Flask, request, jsonify, session
from pyngrok import ngrok
import threading

app = Flask(__name__)
app.secret_key = "3e525f1b5e8aac2ab312793d4201ea14v"  # your secret key

@app.route('/')
def index():
    return "Chatbot API running..."


@app.route('/chat', methods=['POST'])
def chat():
    # Temporarily allow unauthenticated for testing; later add 2FA checks
    data = request.json
    user_query = data.get('message', '')
    if not user_query:
        return jsonify({"error": "No message provided"}), 400

    response = get_best_answer(user_query)  # Your chatbot handler function
    return jsonify({"response": response})

def start_ngrok():
    public_url = ngrok.connect(5000)
    print(f" * ngrok tunnel available at: {public_url}")

threading.Thread(target=start_ngrok).start()

app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


 * ngrok tunnel available at: NgrokTunnel: "https://sporogenous-unsatisfied-ula.ngrok-free.dev" -> "http://localhost:5000"


INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:05:33] "POST / HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:05:47] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:05:53] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:06:40] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:06:41] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:07:34] "POST / HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:08:58] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:10:26] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:10:34] "POST /chat HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [10/Nov/2025 03:10:40] "POST /chat HTTP/1.1" 200 -


In [22]:
from pyngrok import ngrok
ngrok.set_auth_token("33uNM1mtRSXOYOjraapnSurcFKL_4FSVtgDYjKUZEsrVaoucN")  # Put your key as a string here
